# Credit Risk — Feature Engineering

__Objective__

Develop behavioral and financial features based on
the findings from exploratory data analysis.

The engineered features aim to capture:

- Delinquency severity, frequency and recency.
- Historical billing behavior.
- Payment patterns and trends.
- Credit utilization.
- Payment-to-bill relationships.

Original predictors will be preserved to enable
comparisons between baseline and engineered
feature sets.

# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

# 2. Data Loading

In [2]:
# Load the original dataset
dataset = fetch_ucirepo(id=350)

X = dataset.data.features.copy()
y = dataset.data.targets.copy().squeeze()

# Rename variables
column_mapping = {
    "X1": "LIMIT_BAL",
    "X2": "SEX",
    "X3": "EDUCATION",
    "X4": "MARRIAGE",
    "X5": "AGE",
    **{f"X{i+5}": f"PAY_{i}" for i in range(1, 7)},
    **{f"X{i+11}": f"BILL_AMT{i}" for i in range(1, 7)},
    **{f"X{i+17}": f"PAY_AMT{i}" for i in range(1, 7)}
}

X = X.rename(columns=column_mapping)

display(X.head())

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679


# 3. Train-Test Split

A stratified split is used to preserve the original
target distribution in both subsets.

The test set will be reserved for final evaluation.
Data-dependent preprocessing, feature selection
and model optimization will use training data only.

Since target-based EDA was previously conducted
on the full dataset, the test set cannot be
considered a completely untouched holdout.

In [3]:
# Perform stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Validate dataset sizes and target distribution
split_summary = pd.DataFrame({
    "number_of_clients": [
        len(y),
        len(y_train),
        len(y_test)
    ],
    "default_rate": [
        y.mean() * 100,
        y_train.mean() * 100,
        y_test.mean() * 100
    ]
}, index=["Original", "Train", "Test"])

display(split_summary.round(2))

,number_of_clients,default_rate
Original,30000,22.12
Train,24000,22.12
Test,6000,22.12


# 4. Delinquency Features

Delinquency features summarize historical payment
status in terms of severity, frequency, persistence
and recent changes.

Special payment status codes (-2, -1, 0) are treated
as zero months of delinquency for these calculations.

Original payment status variables are preserved.

__General Assumptions__

Special payment status codes (-2, -1, 0) are treated
as zero months of delinquency exclusively for the
calculation of these engineered features.

For each period, delinquency severity is defined as:

$$
D_i = \max(PAY_i, 0)
$$

Original payment status variables are preserved to
retain the information contained in special codes.

PAY_1 represents the most recent period, while
PAY_6 represents the oldest observed period.

__4.1. MAX_DELAY__

**Definition**

Represents the maximum delinquency severity observed
across the six historical periods.

**Calculation**

$$
MAX\_DELAY = \max_{i=1,\ldots,6}(D_i)
$$

**EDA Evidence**

Payment status exhibited a strong association with
credit default.

In the most recent period, default rates increased
from 12.81% for status code 0 to 33.95% for code 1
and 69.14% for code 2.

**Business Rationale**

The maximum historical delay may capture severe
episodes of financial distress that are not
adequately represented by average behavior.

Clients who have experienced severe delinquency
may exhibit elevated future credit risk.

**Limitations**

- Does not distinguish recent from older delays.
- Does not capture delinquency frequency.
- Rare high-severity categories exhibited unstable
  default rate estimates during EDA.


__4.2. AVG_DELAY__

**Definition**

Represents the average delinquency severity across
all six observed periods, including periods without
positive delinquency codes.

**Calculation**

$$
AVG\_DELAY = \frac{1}{6}\sum_{i=1}^{6}D_i
$$

**EDA Evidence**

Positive delinquency codes were consistently
associated with elevated default rates across
historical payment periods.

Cramér's V analysis also identified relevant
associations between payment status variables.

**Business Rationale**

Average delinquency severity summarizes the
client's overall historical payment behavior.

Unlike MAX_DELAY, this feature considers the
entire observation window and may distinguish
persistent delinquency from isolated episodes.

**Limitations**

- Different delinquency histories may produce
  identical averages.
- Recent and older periods receive equal weight.
- Special payment status distinctions are lost
  in the aggregation.


__4.3. DELAY_COUNT__

**Definition**

Represents the number of observed periods in which
the client exhibited a positive delinquency code.

**Calculation**

$$
DELAY\_COUNT =
\sum_{i=1}^{6}\mathbb{1}(D_i > 0)
$$

**EDA Evidence**

Positive delinquency codes were associated with
higher default rates across multiple periods.

Associations between historical payment status
variables suggest persistence in payment behavior.

**Business Rationale**

Repeated delinquency may indicate recurring
financial difficulties or persistent problems
meeting payment obligations.

This feature distinguishes isolated delinquency
events from recurring delinquency behavior.

**Limitations**

- Does not distinguish mild from severe delays.
- Does not capture the temporal ordering of delays.
- Different delinquency patterns may produce
  identical counts.


__4.4. CONSECUTIVE_DELAYS__

**Definition**

Represents the number of consecutive periods with
positive delinquency codes, starting from the
most recent observed period.

**Calculation**

$$
CONSECUTIVE\_DELAYS =
\sum_{i=1}^{6}
\prod_{j=1}^{i}\mathbb{1}(D_j > 0)
$$

**EDA Evidence**

Recent payment status exhibited particularly
strong associations with credit default.

Cramér's V analysis identified stronger associations
between consecutive payment periods, with associations
generally decreasing as temporal distance increased.

**Business Rationale**

Uninterrupted recent delinquency may indicate
ongoing financial distress rather than an isolated
historical payment problem.

This feature combines delinquency persistence
and recency into a single behavioral indicator.

**Limitations**

- Ignores delinquency episodes occurring before
  the first non-delinquent period.
- Does not capture delinquency severity.
- May exhibit redundancy with DELAY_COUNT and
  the most recent payment status.


__4.5. DELAY_TREND__

**Definition**

Represents the difference between average delinquency
severity in the three most recent periods and
the three oldest periods.

**Calculation**

$$
DELAY\_TREND =
\frac{D_1+D_2+D_3}{3}
-
\frac{D_4+D_5+D_6}{3}
$$

**EDA Evidence**

Recent payment status exhibited stronger
associations with default than older payment
status variables.

Associations between payment status variables
also decreased with temporal distance.

**Business Rationale**

Changes in delinquency severity may indicate
improving or deteriorating financial behavior.

Positive values indicate higher recent delinquency
severity, while negative values indicate lower
recent delinquency severity.

**Limitations**

- Represents a difference between period averages,
  rather than a fitted temporal trend.
- The three-period window is a modeling choice.
- Short-term fluctuations may be concealed by
  averaging.
- Predictive relevance must be empirically validated.

In [4]:
def add_delinquency_features(X):

    X = X.copy()

    delay_columns = [f"PAY_{i}" for i in range(1, 7)]

    # Convert non-positive status codes to zero
    delays = X[delay_columns].clip(lower=0)

    # Maximum delinquency severity
    X["MAX_DELAY"] = delays.max(axis=1)

    # Average delinquency severity
    X["AVG_DELAY"] = delays.mean(axis=1)

    # Number of periods with delinquency
    X["DELAY_COUNT"] = (delays > 0).sum(axis=1)

    # Consecutive delinquent periods from most recent
    X["CONSECUTIVE_DELAYS"] = (
        (delays > 0).cumprod(axis=1).sum(axis=1)
    )

    # Recent vs. older delinquency severity
    X["DELAY_TREND"] = (
        delays.iloc[:, :3].mean(axis=1)
        - delays.iloc[:, 3:].mean(axis=1)
    )

    return X

In [5]:
X_train_fe = add_delinquency_features(X_train)

new_columns = [
    "MAX_DELAY",
    "AVG_DELAY",
    "DELAY_COUNT",
    "CONSECUTIVE_DELAYS",
    "DELAY_TREND"
]

display(X_train_fe.head(10))

display(X_train_fe[new_columns].describe().round(2))

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,MAX_DELAY,AVG_DELAY,DELAY_COUNT,CONSECUTIVE_DELAYS,DELAY_TREND
22788,160000,2,2,2,33,2,2,3,2,0,0,161771,172632,168541,164310,162681,163005,15000,0,0,6100,12300,6100,3,1.500000,4,4,1.666667
29006,150000,2,1,2,34,1,-1,-1,-2,-2,-2,0,53,0,0,0,0,53,0,0,0,0,0,1,0.166667,1,1,0.333333
16950,10000,1,2,1,50,1,2,0,0,0,0,10171,8150,8446,8067,8227,8400,2,1281,1134,294,305,1000,2,0.500000,2,2,1.000000
22280,220000,2,1,2,29,0,0,0,0,0,0,206879,211037,215139,218513,131660,134346,9100,9000,7887,4800,4900,6000,0,0.000000,0,0,0.000000
11346,310000,2,1,2,32,1,-2,-1,0,0,-2,0,0,326,326,-235,-235,0,326,0,0,0,1200,1,0.166667,1,1,0.333333
4102,240000,2,1,2,36,-1,-1,2,0,-1,2,5367,6959,6649,5149,13750,5777,6959,15,0,7013,0,726,2,0.666667,2,0,0.000000
28654,160000,2,3,2,33,-1,2,2,2,2,-2,18195,755,44255,43228,-772,-772,0,43500,0,0,0,3000,2,1.333333,4,0,0.000000
26150,50000,2,2,1,34,3,3,2,2,2,2,47904,48240,48602,46289,44837,44535,2000,2001,499,2000,2000,1500,3,2.333333,6,6,0.666667
15091,360000,1,1,1,32,0,0,0,0,0,0,120533,124530,128518,134368,135626,134864,6000,6000,8000,4918,5000,3300,0,0.000000,0,0,0.000000
6266,320000,1,2,2,29,0,0,0,0,0,0,324693,321110,325024,322246,297637,301575,11650,12500,11900,10503,10886,10040,0,0.000000,0,0,0.000000


,MAX_DELAY,AVG_DELAY,DELAY_COUNT,CONSECUTIVE_DELAYS,DELAY_TREND
count,24000.00,24000.00,24000.00,24000.00,24000.00
mean,0.68,0.28,0.84,0.62,0.09
std,1.07,0.60,1.55,1.46,0.57
min,0.00,0.00,0.00,0.00,-4.67
25%,0.00,0.00,0.00,0.00,0.00
50%,0.00,0.00,0.00,0.00,0.00
75%,2.00,0.33,1.00,0.00,0.00
max,8.00,6.00,6.00,6.00,3.33


In [6]:
# Validate engineered delinquency features
assert X_train_fe[new_columns].notna().all().all()

assert X_train_fe["MAX_DELAY"].between(0, 9).all()

assert X_train_fe["AVG_DELAY"].between(0, 9).all()

assert X_train_fe["DELAY_COUNT"].between(0, 6).all()

assert X_train_fe["CONSECUTIVE_DELAYS"].between(0, 6).all()

assert X_train_fe["DELAY_TREND"].between(-9, 9).all()

assert (
    X_train_fe["CONSECUTIVE_DELAYS"]
    <= X_train_fe["DELAY_COUNT"]
).all()

assert (
    X_train_fe["AVG_DELAY"]
    <= X_train_fe["MAX_DELAY"]
).all()

print("All delinquency feature validation checks passed.")

All delinquency feature validation checks passed.


__Delinquency Features — Key Findings__

- All five engineered features contain 24,000
  non-missing observations.
- At least half of the training population exhibits
  no positive delinquency codes across the six periods.
- Recent consecutive delinquency is concentrated
  among a minority of clients.
- Delinquency trend captures both increases and
  decreases in average delinquency severity.
- Predictive contribution and potential redundancy
  will be evaluated during feature selection.

# 5. Billing Features

Billing features summarize historical outstanding
balances in terms of magnitude, variability,
recent changes and special balance conditions.

Negative and zero bill amounts are preserved,
as identified during exploratory data analysis.

__General Assumptions__

Negative and zero bill amounts are preserved because
they may represent distinct financial conditions.

Original billing variables are retained to enable
comparisons between original and aggregated
feature sets.

BILL_AMT1 represents the most recent billing period,
while BILL_AMT6 represents the oldest observed period.

__5.1. AVG_BILL__

**Definition**

Represents the average bill amount across the six
historical billing periods.

**Calculation**

$$
AVG\_BILL =
\frac{1}{6}\sum_{i=1}^{6}BILL\_AMT_i
$$

**EDA Evidence**

Bill amount variables exhibited strong positive
Pearson correlations, ranging from 0.80 to 0.95.

Strong correlations were also observed using
Spearman correlation.

These findings suggest substantial redundancy
and persistence across billing periods.

**Business Rationale**

Average bill amount provides a summarized
representation of historical financial exposure.

It may reduce the influence of isolated billing
fluctuations and serve as an alternative to
multiple highly correlated billing variables.

**Limitations**

- May conceal relevant short-term fluctuations.
- Recent and older periods receive equal weight.
- Negative bill amounts may offset positive balances.
- Does not account for differences in credit limits.


__5.2. MAX_BILL__

**Definition**

Represents the highest bill amount observed across
the six historical billing periods.

**Calculation**

$$
MAX\_BILL =
\max_{i=1,\ldots,6}(BILL\_AMT_i)
$$

**EDA Evidence**

Billing variables exhibited strongly right-skewed
distributions with substantial extreme values.

Strong correlations between billing periods
suggest that aggregated representations may
capture complementary aspects of historical
billing behavior.

**Business Rationale**

The maximum bill amount captures the highest
observed billing exposure.

This feature may identify periods of unusually
high credit usage that would otherwise be
concealed by average bill amounts.

**Limitations**

- Sensitive to extreme observations.
- Does not indicate whether high exposure
  was persistent or temporary.
- Does not account for the client's credit limit.
- High absolute bill amounts do not necessarily
  indicate elevated credit risk.


__5.3. BILL_STD__

**Definition**

Represents the standard deviation of bill amounts
across the six observed billing periods.

**Calculation**

$$
BILL\_STD =
\sqrt{
\frac{1}{6}
\sum_{i=1}^{6}
(BILL\_AMT_i-AVG\_BILL)^2
}
$$

**EDA Evidence**

Billing variables exhibited strong correlations
and substantial differences in monetary magnitude.

These findings motivated the investigation of
aggregated features that describe aspects of
billing behavior beyond average exposure.

**Business Rationale**

Billing variability may capture changes in
credit usage or financial behavior over time.

Clients with similar average bill amounts may
exhibit substantially different levels of
billing stability.

**Limitations**

- Sensitive to extreme bill amounts.
- Does not distinguish increasing from decreasing
  billing patterns.
- Absolute variability may depend on the client's
  overall financial exposure.
- Population standard deviation is used to describe
  the six observed periods (ddof=0).


__5.4. BILL_TREND__

**Definition**

Represents the difference between the average bill
amount in the three most recent periods and the
three oldest periods.

**Calculation**

$$
BILL\_TREND =
\frac{BILL\_AMT1+BILL\_AMT2+BILL\_AMT3}{3}
-
\frac{BILL\_AMT4+BILL\_AMT5+BILL\_AMT6}{3}
$$

**EDA Evidence**

Correlations between bill amounts generally
decreased as the temporal distance between
billing periods increased.

Although bill amounts exhibited substantial
persistence, historical periods were not
perfectly correlated.

**Business Rationale**

Changes in billing amounts may capture increasing
or decreasing financial exposure.

Positive values indicate higher recent bill amounts,
while negative values indicate lower recent
bill amounts.

This feature may capture behavioral changes that
are not represented by historical averages.

**Limitations**

- Represents a difference between period averages,
  rather than a fitted temporal trend.
- Increasing bill amounts do not necessarily
  indicate financial distress.
- Sensitive to extreme monetary values.
- The three-period window is a modeling choice.


__5.5. ZERO_BILL_COUNT__

**Definition**

Represents the number of historical periods with
bill amounts equal to zero.

**Calculation**

$$
ZERO\_BILL\_COUNT =
\sum_{i=1}^{6}\mathbb{1}(BILL\_AMT_i=0)
$$

**EDA Evidence**

In the most recent billing period, clients with
zero bill amounts exhibited a default rate of
26.59%, compared with 21.87% for positive bill
amounts and 18.47% for negative bill amounts.

This suggests that zero bill amounts may carry
distinct predictive information.

**Business Rationale**

Repeated zero bill amounts may represent specific
patterns of credit usage, account inactivity
or other financial conditions.

Explicitly counting these occurrences may help
the model distinguish clients with different
historical billing patterns.

**Limitations**

- The underlying reasons for zero bill amounts
  are not available in the dataset.
- Zero balances do not necessarily indicate
  financial distress.
- The observed association for the most recent
  period does not establish the predictive
  relevance of the historical count.


__5.6. NEGATIVE_BILL_COUNT__

**Definition**

Represents the number of historical periods with
negative bill amounts.

**Calculation**

$$
NEGATIVE\_BILL\_COUNT =
\sum_{i=1}^{6}\mathbb{1}(BILL\_AMT_i<0)
$$

**EDA Evidence**

Negative bill amounts were observed across all
six billing periods.

Clients with negative bill amounts generally
exhibited lower default rates than clients
with non-negative bill amounts.

However, the relationship between the frequency
of negative bills and default was not monotonic.

**Business Rationale**

Negative bill amounts may represent credit
balances, refunds, overpayments or other
account adjustments.

Counting their occurrences may capture recurring
financial conditions that differ from conventional
positive billing behavior.

**Limitations**

- The exact origin of negative balances cannot
  be determined from the available data.
- Groups with frequent negative bills contain
  relatively few observations.
- No monotonic relationship with default should
  be assumed.
- Predictive contribution must be empirically
  validated.

In [7]:
def add_billing_features(X):

    X = X.copy()

    bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]

    bills = X[bill_columns]

    # Average bill amount
    X["AVG_BILL"] = bills.mean(axis=1)

    # Maximum bill amount
    X["MAX_BILL"] = bills.max(axis=1)

    # Bill amount variability
    X["BILL_STD"] = bills.std(axis=1, ddof=0)

    # Recent vs. older bill amounts
    X["BILL_TREND"] = (
        bills.iloc[:, :3].mean(axis=1)
        - bills.iloc[:, 3:].mean(axis=1)
    )

    # Number of zero bill amounts
    X["ZERO_BILL_COUNT"] = (bills == 0).sum(axis=1)

    # Number of negative bill amounts
    X["NEGATIVE_BILL_COUNT"] = (bills < 0).sum(axis=1)

    return X

In [8]:
X_train_fe = add_billing_features(X_train_fe)

billing_features = [
    "AVG_BILL",
    "MAX_BILL",
    "BILL_STD",
    "BILL_TREND",
    "ZERO_BILL_COUNT",
    "NEGATIVE_BILL_COUNT"
]

display(X_train_fe[billing_features].describe().round(2))

print(X_train_fe.shape)

,AVG_BILL,MAX_BILL,BILL_STD,BILL_TREND,ZERO_BILL_COUNT,NEGATIVE_BILL_COUNT
count,24000.00,24000.00,24000.00,24000.00,24000.00,24000.00
mean,44828.62,60355.78,10992.18,8325.87,0.60,0.13
std,63020.66,77479.37,18097.66,29552.32,1.45,0.61
min,-56043.17,-4300.00,0.00,-291275.00,0.00,0.00
25%,4802.54,10099.75,1424.69,-1912.75,0.00,0.00
50%,21044.83,31236.00,4188.09,639.17,0.00,0.00
75%,56893.25,79557.25,13174.32,12684.17,0.00,0.00
max,877313.83,983931.00,308946.17,432428.33,6.00,6.00


(24000, 34)


In [9]:
# Validate engineered billing features
assert np.isfinite(X_train_fe[billing_features]).all().all()

assert (X_train_fe["BILL_STD"] >= 0).all()

assert (
    X_train_fe["AVG_BILL"]
    <= X_train_fe["MAX_BILL"]
).all()

assert X_train_fe["ZERO_BILL_COUNT"].between(0, 6).all()

assert X_train_fe["NEGATIVE_BILL_COUNT"].between(0, 6).all()

assert (
    X_train_fe["ZERO_BILL_COUNT"]
    + X_train_fe["NEGATIVE_BILL_COUNT"]
    <= 6
).all()

print("All billing feature validation checks passed.")

All billing feature validation checks passed.


__Billing Features — Key Findings__

- All six engineered billing features contain
  24,000 non-missing observations.
- Average and maximum bill amounts retain
  positively skewed distributions.
- Negative bill amounts remain represented
  in aggregated features.
- Billing trends capture both increases and
  decreases in recent bill amounts.
- Zero and negative bill counts remain within
  their expected ranges.
- Aggregated billing features will be evaluated
  as potential alternatives to correlated
  original billing variables.

# 6. Payment Features

Payment features summarize historical payment
behavior in terms of magnitude, frequency,
recency, persistence and recent changes.

Zero payments are explicitly represented due
to their observed association with credit default.

However, zero payments do not necessarily
indicate delinquency.

__General Assumptions__

Zero payments are explicitly represented due
to their observed association with credit default.

However, zero payments do not necessarily indicate
delinquency, as clients may have no outstanding
payment obligations.

Original payment amount variables are preserved.

PAY_AMT1 represents the most recent payment
period, while PAY_AMT6 represents the oldest
observed period.

__6.1. AVG_PAYMENT__

**Definition**

Represents the average payment amount across the
six historical payment periods.

**Calculation**

$$
AVG\_PAYMENT =
\frac{1}{6}\sum_{i=1}^{6}PAY\_AMT_i
$$

**EDA Evidence**

Higher positive payment amounts were generally
associated with lower default rates across
all six periods.

Payment amount variables exhibited weak Pearson
correlations but moderate Spearman correlations.

**Business Rationale**

Average payment amount summarizes the client's
historical repayment magnitude.

It may provide a more stable representation
of repayment behavior than individual monthly
payment amounts.

Including zero payments allows the feature to
reflect both payment magnitude and periods
without payments.

**Limitations**

- Sensitive to extreme payment amounts.
- Does not account for outstanding bill amounts.
- May conceal relevant recent changes.
- Different payment histories may produce
  identical averages.


__6.2. ZERO_PAYMENT_COUNT__

**Definition**

Represents the number of historical periods in
which the recorded payment amount was zero.

**Calculation**

$$
ZERO\_PAYMENT\_COUNT =
\sum_{i=1}^{6}\mathbb{1}(PAY\_AMT_i=0)
$$

**EDA Evidence**

Zero payments were consistently associated with
higher default rates across all six periods.

Default rates among clients with zero payments
ranged from 28.97% to 35.95%, compared with
approximately 19% to 20% among clients with
positive payments.

**Business Rationale**

Repeated periods without payments may indicate
persistent difficulties meeting financial
obligations.

This feature captures the frequency of zero
payments independently of their monetary magnitude.

**Limitations**

- Zero payments do not necessarily indicate
  missed contractual obligations.
- Does not distinguish recent from older
  zero payments.
- Does not capture the amount that was due.
- Different temporal patterns may produce
  identical counts.


__6.3. LAST_PAYMENT_ZERO__

**Definition**

Binary indicator identifying whether the most
recent recorded payment amount was zero.

**Calculation**

$$
LAST\_PAYMENT\_ZERO =
\mathbb{1}(PAY\_AMT1=0)
$$

**EDA Evidence**

The most recent zero payment exhibited the
strongest observed association with default
among the six payment periods.

Clients with zero payments in the most recent
period exhibited a default rate of 35.95%,
compared with 19.19% for clients with positive
payments.

This represents a difference of 16.76 percentage
points.

**Business Rationale**

Recent payment behavior may provide a more
immediate indication of the client's current
financial condition.

An explicit binary indicator may help models
capture the distinct association between zero
payments and credit default.

**Limitations**

- Derived directly from PAY_AMT1 and therefore
  does not introduce new raw information.
- Can also be derived from
  CONSECUTIVE_ZERO_PAYMENTS.
- Does not distinguish absence of payment
  obligations from missed payments.
- Potential redundancy must be evaluated during
  feature selection.


__6.4. CONSECUTIVE_ZERO_PAYMENTS__

**Definition**

Represents the number of consecutive periods with
zero payment amounts, starting from the most
recent observed period.

**Calculation**

$$
CONSECUTIVE\_ZERO\_PAYMENTS =
\sum_{i=1}^{6}
\prod_{j=1}^{i}
\mathbb{1}(PAY\_AMT_j=0)
$$

**EDA Evidence**

Zero payments were consistently associated with
higher default rates across historical periods.

The difference in default rates between zero
and positive payments increased for more
recent periods, ranging from 9.00 percentage
points in the oldest period to 16.76 percentage
points in the most recent period.

**Business Rationale**

Consecutive recent zero payments may indicate
ongoing financial difficulties rather than
isolated historical events.

This feature combines payment frequency,
persistence and recency.

**Limitations**

- Ignores zero payments occurring before the
  first positive payment.
- Does not account for outstanding bill amounts.
- May exhibit redundancy with ZERO_PAYMENT_COUNT
  and LAST_PAYMENT_ZERO.
- The predictive relevance of consecutive zero
  payments has not yet been directly established.


__6.5. PAYMENT_TREND__

**Definition**

Represents the difference between average payment
amounts in the three most recent periods and
the three oldest periods.

**Calculation**

$$
PAYMENT\_TREND =
\frac{PAY\_AMT1+PAY\_AMT2+PAY\_AMT3}{3}
-
\frac{PAY\_AMT4+PAY\_AMT5+PAY\_AMT6}{3}
$$

**EDA Evidence**

Higher positive payment amounts were generally
associated with lower default rates.

The association between zero payments and default
was stronger in more recent periods.

Payment amounts also exhibited moderate rank
correlations across historical periods.

**Business Rationale**

Changes in repayment magnitude may capture
improvements or deterioration in financial
behavior.

Positive values indicate higher recent payment
amounts, while negative values indicate lower
recent payment amounts.

This feature may provide information about
behavioral changes that historical averages
cannot capture.

**Limitations**

- Represents a difference between period averages,
  rather than a fitted temporal trend.
- Lower payments may reflect lower outstanding
  balances rather than financial deterioration.
- Sensitive to extreme payment amounts.
- Does not account for changes in payment
  obligations.
- The three-period window is a modeling choice.

In [11]:
def add_payment_features(X):

    X = X.copy()

    payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

    payments = X[payment_columns]

    # Average payment amount
    X["AVG_PAYMENT"] = payments.mean(axis=1)

    # Number of zero payments
    X["ZERO_PAYMENT_COUNT"] = (payments == 0).sum(axis=1)

    # Most recent payment is zero
    X["LAST_PAYMENT_ZERO"] = (
        payments["PAY_AMT1"] == 0
    ).astype(int)

    # Consecutive zero payments from most recent
    X["CONSECUTIVE_ZERO_PAYMENTS"] = (
        (payments == 0).cumprod(axis=1).sum(axis=1)
    )

    # Recent vs. older payment amounts
    X["PAYMENT_TREND"] = (
        payments.iloc[:, :3].mean(axis=1)
        - payments.iloc[:, 3:].mean(axis=1)
    )

    return X

In [12]:
X_train_fe = add_payment_features(X_train_fe)

payment_features = [
    "AVG_PAYMENT",
    "ZERO_PAYMENT_COUNT",
    "LAST_PAYMENT_ZERO",
    "CONSECUTIVE_ZERO_PAYMENTS",
    "PAYMENT_TREND"
]

display(X_train_fe[payment_features].describe().round(2))

print(X_train_fe.shape)

,AVG_PAYMENT,ZERO_PAYMENT_COUNT,LAST_PAYMENT_ZERO,CONSECUTIVE_ZERO_PAYMENTS,PAYMENT_TREND
count,24000.00,24000.00,24000.00,24000.00,24000.00
mean,5251.58,1.23,0.17,0.49,643.04
std,9850.57,1.72,0.38,1.40,13176.98
min,0.00,0.00,0.00,0.00,-177931.33
25%,1116.67,0.00,0.00,0.00,-316.67
50%,2405.67,0.00,0.00,0.00,400.00
75%,5642.71,2.00,0.00,0.00,1532.00
max,627344.33,6.00,1.00,6.00,730688.67


(24000, 39)


In [13]:
# Validate engineered payment features
assert np.isfinite(X_train_fe[payment_features]).all().all()

assert (X_train_fe["AVG_PAYMENT"] >= 0).all()

assert X_train_fe["ZERO_PAYMENT_COUNT"].between(0, 6).all()

assert X_train_fe["LAST_PAYMENT_ZERO"].isin([0, 1]).all()

assert X_train_fe["CONSECUTIVE_ZERO_PAYMENTS"].between(0, 6).all()

assert (
    X_train_fe["CONSECUTIVE_ZERO_PAYMENTS"]
    <= X_train_fe["ZERO_PAYMENT_COUNT"]
).all()

assert (
    X_train_fe["LAST_PAYMENT_ZERO"]
    == (X_train_fe["CONSECUTIVE_ZERO_PAYMENTS"] > 0).astype(int)
).all()

print("All payment feature validation checks passed.")

All payment feature validation checks passed.


__Payment Features — Key Findings__

- All five engineered payment features contain
  24,000 non-missing observations.
- Average payment amounts retain a strongly
  right-skewed distribution.
- Clients exhibit an average of 1.23 periods
  with zero payments.
- Approximately 17% of clients have a zero
  payment in the most recent period.
- Payment trend captures both increases and
  decreases in recent payment amounts.
- Potential redundancy between payment indicators
  will be investigated during feature selection.

# 7. Financial Ratio Features

Financial ratio features combine billing, payment
and credit limit information to capture relative
financial behavior rather than absolute monetary
amounts.

__7.1. CREDIT_UTILIZATION__

**Definition**

Represents the proportion of the reported credit
limit occupied by the most recent bill amount.

**Calculation**

$$
CREDIT\_UTILIZATION =
\frac{BILL\_AMT1}{LIMIT\_BAL}
$$

**EDA Evidence**

Credit limit exhibited a strong inverse association
with default, with default rates decreasing from
35.85% in the lowest credit limit group to 11.87%
in the highest group.

In contrast, positive bill amounts exhibited a
weaker and non-monotonic relationship with default.

**Business Rationale**

The same outstanding balance may represent
different levels of financial exposure depending
on the client's available credit limit.

Higher credit utilization may indicate greater
financial pressure or dependence on credit.

Combining bill amounts and credit limits may
therefore provide more meaningful predictive
information than absolute bill amounts alone.

**Limitations**

- Bill amounts are used as a proxy for outstanding
  credit utilization.
- Negative values and ratios above 1 are preserved.
- Non-positive credit limits produce undefined ratios.
- Predictive contribution must be empirically validated.


__7.2. AVG_CREDIT_UTILIZATION__

**Definition**

Represents the average bill amount across the six
observed periods relative to the reported credit limit.

**Calculation**

$$
AVG\_CREDIT\_UTILIZATION =
\frac{\frac{1}{6}\sum_{i=1}^{6} BILL\_AMT_i}
{LIMIT\_BAL}
$$

**EDA Evidence**

Billing variables exhibited strong correlations,
with Pearson coefficients ranging from 0.80 to 0.95.

This indicates substantial persistence in billing
amounts across historical periods.

Additionally, absolute bill amounts exhibited
relatively weak standalone associations with default.

**Business Rationale**

While CREDIT_UTILIZATION captures recent financial
exposure, AVG_CREDIT_UTILIZATION summarizes
historical exposure relative to the credit limit.

This feature may help distinguish persistent credit
utilization from temporary increases in billing amounts.

It may also provide a more stable representation
of historical financial exposure.

**Limitations**

- The reported credit limit is assumed to be
  representative of all six periods.
- Historical credit limit changes are unavailable.
- Averaging may conceal relevant short-term changes.
- Potential redundancy with CREDIT_UTILIZATION
  must be evaluated.


__7.3. PAYMENT_TO_BILL_RATIO__

**Definition**

Represents the aggregate proportion of eligible
historical bill amounts covered by subsequent
payments.

**Calculation**

$$
PAYMENT\_TO\_BILL\_RATIO =
\frac{\sum_{i \in S} PAY\_AMT_i}
{\sum_{i \in S} BILL\_AMT_{i+1}}
$$

Where S contains periods from 1 to 5 for which
the corresponding previous bill amount is positive.

**EDA Evidence**

Zero payments were consistently associated with
higher default rates across all six periods.

In the most recent period, clients with zero
payments exhibited a default rate of 35.95%,
compared with 19.19% for clients with positive
payments.

Higher positive payment amounts were generally
associated with lower default rates.

Spearman correlation analysis also identified
relatively strong associations between payments
and previous billing periods, with coefficients
ranging approximately from 0.63 to 0.67.

**Business Rationale**

Absolute payment amounts do not necessarily
reflect a client's ability to meet financial
obligations.

A payment of 5,000 may represent full repayment
for one client but only a small fraction of the
outstanding balance for another.

Relating payments to corresponding bill amounts
may provide a more informative representation
of repayment behavior.

The ratio of aggregated amounts is used instead
of averaging individual monthly ratios to reduce
the disproportionate influence of small monthly
denominators.

**Limitations**

- A one-period lag between bills and payments
  is assumed but cannot be fully verified.
- PAY_AMT6 is excluded because the corresponding
  previous bill amount is unavailable.
- Only pairs with positive bill amounts are included.
- Ratios are undefined when no eligible positive
  bill amounts exist.
- Small aggregate denominators may produce
  extreme ratios.
- Ratios above 1 are preserved.
- This feature does not necessarily represent
  the proportion of contractual obligations fulfilled.

In [14]:
def add_financial_ratio_features(X):

    X = X.copy()

    bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]

    # Avoid division by non-positive credit limits
    credit_limit = X["LIMIT_BAL"].where(X["LIMIT_BAL"] > 0)

    # Most recent credit utilization
    X["CREDIT_UTILIZATION"] = (
        X["BILL_AMT1"] / credit_limit
    )

    # Average credit utilization
    X["AVG_CREDIT_UTILIZATION"] = (
        X[bill_columns].mean(axis=1) / credit_limit
    )

    # Align payments with previous billing periods
    bills = X[
        [f"BILL_AMT{i}" for i in range(2, 7)]
    ].to_numpy()

    payments = X[
        [f"PAY_AMT{i}" for i in range(1, 6)]
    ].to_numpy()

    # Identify eligible positive bill amounts
    valid_pairs = bills > 0

    # Aggregate matched bills and payments
    total_bills = np.where(
        valid_pairs, bills, 0
    ).sum(axis=1)

    total_payments = np.where(
        valid_pairs, payments, 0
    ).sum(axis=1)

    # Calculate payment-to-bill ratio safely
    X["PAYMENT_TO_BILL_RATIO"] = np.divide(
        total_payments,
        total_bills,
        out=np.full(len(X), np.nan),
        where=total_bills > 0
    )

    return X

In [15]:
X_train_fe = add_financial_ratio_features(X_train_fe)

ratio_features = [
    "CREDIT_UTILIZATION",
    "AVG_CREDIT_UTILIZATION",
    "PAYMENT_TO_BILL_RATIO"
]

display(X_train_fe[ratio_features].describe().round(4))

display(X_train_fe[ratio_features].isna().sum())

print(X_train_fe.shape)

,CREDIT_UTILIZATION,AVG_CREDIT_UTILIZATION,PAYMENT_TO_BILL_RATIO
count,24000.0000,24000.0000,22883.0000
mean,0.4233,0.3722,0.3869
std,0.4104,0.3511,1.2357
min,-0.6199,-0.2326,0.0000
25%,0.0218,0.0303,0.0458
50%,0.3149,0.2842,0.1048
75%,0.8288,0.6859,0.7722
max,6.4553,5.3643,129.7051


CREDIT_UTILIZATION           0
AVG_CREDIT_UTILIZATION       0
PAYMENT_TO_BILL_RATIO     1117
dtype: int64

(24000, 42)


__Financial Ratio Features — Extreme Value Analysis__

In [16]:
display(
    X_train_fe[ratio_features].quantile(
        [0.90, 0.95, 0.99, 0.995, 0.999, 1.0]
    ).round(4)
)

,CREDIT_UTILIZATION,AVG_CREDIT_UTILIZATION,PAYMENT_TO_BILL_RATIO
0.900,0.9814,0.8898,1.0003
0.950,1.0128,0.9590,1.0053
0.990,1.2335,1.0481,1.2586
0.995,1.5167,1.1421,1.6855
0.999,2.1037,1.5587,4.8378
1.000,6.4553,5.3643,129.7051


In [17]:
inspection_columns = (
    ["LIMIT_BAL"]
    + [f"BILL_AMT{i}" for i in range(2, 7)]
    + [f"PAY_AMT{i}" for i in range(1, 6)]
    + ["PAYMENT_TO_BILL_RATIO"]
)

display(
    X_train_fe.nlargest(5, "PAYMENT_TO_BILL_RATIO")
    [inspection_columns]
)

,LIMIT_BAL,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAYMENT_TO_BILL_RATIO
28193,160000,-520,0,-390,780,0,380,2000,0,101170,0,129.705128
389,260000,291,582,291,0,291,291,582,0,0,130291,90.147079
21348,70000,400,2,2,2,2,10400,10002,0,0,0,50.004902
24089,40000,-167,-5187,-187,-187,113,167,4982,5000,0,3335,29.513274
5877,280000,-5,-5,8164,0,1133,0,0,8169,0,231133,25.739701


__Financial Ratio Features — Extreme Value Analysis__

- Approximately 99% of defined payment-to-bill
  ratios are below or equal to 1.2586.
- Extreme ratios are associated with relatively
  small eligible billing amounts and large payments.
- The inspected extreme values are consistent
  with the implemented calculation.
- The ratio should be interpreted as a proxy
  for payment behavior rather than an exact
  measure of debt repayment.
- Original values and undefined ratios are preserved.
- Logarithmic transformation will be considered
  during model-specific preprocessing.

# 8. Reusable Feature Engineering Pipeline

In [19]:
def build_features(X):

    X = add_delinquency_features(X)

    X = add_billing_features(X)

    X = add_payment_features(X)

    X = add_financial_ratio_features(X)

    return X

__8.1. Pipeline Validation__

In [20]:
X_train_pipeline = build_features(X_train)

pd.testing.assert_frame_equal(
    X_train_fe,
    X_train_pipeline
)

print("Feature engineering pipeline validated.")

Feature engineering pipeline validated.


__8.2. Applying Feature Engineering__

In [21]:
X_train_fe = build_features(X_train)

X_test_fe = build_features(X_test)

print("Training dataset:", X_train_fe.shape)

print("Testing dataset:", X_test_fe.shape)

Training dataset: (24000, 42)
Testing dataset: (6000, 42)


# 9. Final Feature Validation

In [22]:
# Validate dataset dimensions
assert X_train_fe.shape == (24000, 42)
assert X_test_fe.shape == (6000, 42)

# Validate column consistency
assert X_train_fe.columns.equals(X_test_fe.columns)

# Validate index preservation
assert X_train_fe.index.equals(y_train.index)
assert X_test_fe.index.equals(y_test.index)

# Validate preservation of original features
pd.testing.assert_frame_equal(
    X_train_fe[X_train.columns],
    X_train
)

pd.testing.assert_frame_equal(
    X_test_fe[X_test.columns],
    X_test
)

# Validate absence of infinite values
assert not np.isinf(X_train_fe).any().any()
assert not np.isinf(X_test_fe).any().any()

print("All final validation checks passed.")

All final validation checks passed.


# 10. Final Conclusions

## Feature Engineering — Final Conclusions

### Engineered Features

A total of 19 features were engineered from
the 23 original predictors:

- 5 delinquency features.
- 6 billing features.
- 5 payment features.
- 3 financial ratio features.

### Methodological Decisions

- Original predictors were preserved.
- Historical behavior was summarized through
  severity, frequency, magnitude and recency.
- Negative and zero financial amounts were preserved.
- Undefined financial ratios were retained.
- No arbitrary outlier removal was performed.
- Feature engineering transformations were
  consolidated into a reusable function.

### Limitations

- Financial ratios are approximations of
  underlying financial behavior.
- Payment-to-bill alignment relies on a
  modeling assumption.
- Potential feature redundancy remains.
- Predictive contribution has not yet
  been established.

### Next Steps

- Evaluate feature redundancy.
- Define model-specific preprocessing.
- Perform feature selection within
  cross-validation.
- Compare original and engineered feature sets.

# 11. Module Integration

The feature engineering functions were extracted
into a reusable Python module to ensure consistency
across training, validation and inference.

In [23]:
import sys
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.feature_engineering import build_features

In [24]:
X_train_module = build_features(X_train)

pd.testing.assert_frame_equal(
    X_train_fe,
    X_train_module
)

print("Feature engineering module validated.")

Feature engineering module validated.
